In [1]:
import json
import traceback
from pathlib import Path

import aiida
from aiida import orm

from tqdm import tqdm

from mc3d_data_utils import verify_optimization_convergence, get_base_details_entry

AIIDA_PROFILE_NAME = "mc3d-pbesol-v1-new"

aiida.load_profile(AIIDA_PROFILE_NAME)

LOG_DST = Path().resolve() / "log"

# delete old log:
if LOG_DST.exists():
    LOG_DST.unlink()

def log(msg):
    print(msg)
    with open(LOG_DST, 'a') as f:
        f.write(str(msg) + '\n')

In [2]:
# list aiida groups
! verdi -p {AIIDA_PROFILE_NAME} group list --all --all-users --count

  PK  Label                             Type string    User                    Node count
----  --------------------------------  -------------  --------------------  ------------
   2  20250828-131437                   core.import    aiida@localhost             684786
   1  structures/final                  core           marnik.bercx@epfl.ch         34198
   4  structures/final/converged        core           aiida@localhost              34111
   5  structures/final/converged_tight  core           aiida@localhost              32430


In [ ]:
# To understand data layout, generate node graph image for a random structure & associated output params.
uuid = "fa0ac19e-e759-438b-b851-c1538b30ef48"
! verdi -p {AIIDA_PROFILE_NAME} node graph generate --process-out --process-in -f svg {uuid} -O graph_cod_{uuid}.svg
uuid = "f8a98ee1-5388-4cc0-9bff-72a4641a5bd1"
! verdi -p {AIIDA_PROFILE_NAME} node graph generate --process-out --process-in -f svg {uuid} -O graph_mpds_{uuid}.svg

# we see that
# * the final structure was generated by a PWCalculation (a vc-relax, if checked)
# * the output parameters are also generated from that, so there is no separate SCF

/home/eimre_k/aiida/aiida-core/src/aiida/orm/nodes/data/code/legacy.py:602: AiidaDeprecationWarning: `Code.get_execname` method is deprecated, use `get_executable` instead. (this will be removed in v3)
  warn_deprecation('`Code.get_execname` method is deprecated, use `get_executable` instead.', version=3)
/home/eimre_k/aiida/aiida-core/src/aiida/orm/nodes/data/code/legacy.py:573: AiidaDeprecationWarning: `Code.is_local` method is deprecated, use a `PortableCode` instance and check the type. (this will be removed in v3)
  warn_deprecation(
/home/eimre_k/aiida/aiida-core/src/aiida/orm/nodes/data/code/legacy.py:524: AiidaDeprecationWarning: `Code.get_remote_exec_path` method is deprecated, use `InstalledCode.filepath_executable` instead. (this will be removed in v3)
  warn_deprecation(
Success: Output written to `graph_cod_fa0ac19e-e759-438b-b851-c1538b30ef48.svg`
/home/eimre_k/aiida/aiida-core/src/aiida/orm/nodes/data/code/legacy.py:602: AiidaDeprecationWarning: `Code.get_execname` metho

In [ ]:
import aiida.plugins

PwBaseWorkChain = aiida.plugins.WorkflowFactory("quantumespresso.pw.base")

qb = (
    orm.QueryBuilder()
    .append(orm.Group, filters={"label": "structures/final/converged"}, tag="group")
    .append(orm.StructureData, with_group="group", project=["*"], tag="structure")
    .append(PwBaseWorkChain, with_outgoing="structure", project=["*"], tag="wc")
    .append(
        orm.ArrayData,
        with_incoming="wc",
        edge_filters={"label": "output_trajectory"},
        project=["*"],
    )
    .append(
        orm.Dict,
        with_incoming="wc",
        edge_filters={"label": "output_parameters"},
        project=["*"],
        tag="parameters",
    )
    .append(orm.CalcJobNode, with_outgoing="parameters", project=["*"], tag="final_opt_calc")
)

count = qb.distinct().count()
print(f"Number of structures: {count}")

Number of structures: 34111


In [ ]:
data = {}

count_successful = 0
count_skipped = 0

for structure, workchain, opt_traj, out_params, final_opt_calc in tqdm(qb.iterall()):
    try:

        if verify_optimization_convergence(opt_traj) is None:
            log(f"SKIPPING: Thresholds not met for workchain {workchain}")
            count_skipped += 1
            continue

        extras = structure.base.extras.all
        mc3d_id = extras.get('mc3d_id')

        if not mc3d_id:
            log(f"SKIPPING: MC3D ID not assigned to {extras['source']}")
            count_skipped += 1
            continue

        details_entry = get_base_details_entry(structure, [out_params])
        details_entry["provenance_links"] = [
            {"label": "Final relax calculation", "uuid": final_opt_calc.uuid},
            {"label": "Final structure", "uuid": structure.uuid},
        ]
        data[mc3d_id] = details_entry

        count_successful += 1

    except Exception as exception:
        log(f"Found exception for {structure}: {exception}")
        traceback.print_exc()
        count_skipped += 1
        continue


log(f"Successful entries {count_successful}")
log(f"Skipped entries {count_skipped}")

OUTPUT_PATH = Path().resolve().parent

with (OUTPUT_PATH / "base.json").open("w") as handle:
    json.dump(data, handle, sort_keys=True, indent=2)

0it [00:00, ?it/s]/home/eimre_k/mcxd/discover-mc3d-data/common-library/mc3d_data_utils/__init__.py:38: AiidaDeprecationWarning: `StructureData.extras` is deprecated, use `StructureData.base.extras.all` instead. (this will be removed in v3)
  extras = structure_node.extras
598it [00:08, 76.73it/s]/home/eimre_k/micromamba/envs/aiida/lib/python3.10/site-packages/pymatgen/core/composition.py:1365: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
6787it [01:30, 81.00it/s]/home/eimre_k/micromamba/envs/aiida/lib/python3.10/site-packages/pymatgen/core/composition.py:1365: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x]

Successful entries 34111
Skipped entries 0
